# MD-GRTN Phase 1 — PEMS08
**Changes:** `d_model=128` | `n_heads=4` | `num_layers=5` | `gru_layers=5` | `seq_len=12` | `batch_size=32` | 2-seq MDAF | GAT+GRU | Huber loss | AdamW | **BackNet pre-training** | **temporal position encoding** | noise augmentation | distance adjacency

**Architecture:** MDAF(BackNet pretrained) → MGRC(GAT+GRU ×5) → STFormer(spatial+temporal pos enc) ×5 → Prediction

**VRAM estimate:** ~11 GB / 15 GB T4  ✓


In [1]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════
# SEED — fixes ALL randomness across sessions
# Run this first, every single session
# ══════════════════════════════════════════════════
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} — results reproducible across sessions ✓')

set_seed()

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

Seed set: 42 — results reproducible across sessions ✓
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


In [2]:
class Config:
    data_path    = "/content/PEMS08.npz"
    num_nodes    = 170
    in_features  = 3
    seq_len      = 12      # paper uses 12
    pred_len     = 12
    feature_idx  = 0
    noise_std    = 10.0    # FIX 1: noisy input → clean output (paper §V-A)
    hour_offset  = 12      # FIX 5: 1 hr ago = 12 × 5-min steps
    train_ratio  = 0.7
    val_ratio    = 0.1
    steps_per_day = 288

    d_model      = 96
    n_heads      = 4
    gat_layers   = 4
    temp_layers  = 5
    dropout      = 0.25

    batch_size   = 64
    lr           = 1e-3    # paper uses 1e-3
    epochs       = 200
    patience     = 30
    weight_decay = 0.01    # FIX 4: paper uses 0.01 (100x stronger)
    best_path    = "mdgrtn_best.pt"

cfg    = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config | d={cfg.d_model} | GAT={cfg.gat_layers} | "
      f"Temp={cfg.temp_layers} | seq={cfg.seq_len} | {device}")


Config | d=96 | GAT=4 | Temp=5 | seq=18 | batch=64 | cuda


In [3]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw["data"].astype(np.float32)
    T, N, F = data.shape
    print(f"Shape: {data.shape}")
    mean_np = data.mean(axis=0)          # (N, F) per-sensor
    std_np  = data.std(axis=0) + 1e-8
    data_clean = (data - mean_np[None]) / std_np[None]
    # Noise in normalised space (paper adds σ=10 to raw flow)
    feat_std = std_np[:, cfg.feature_idx].mean()
    norm_noise = cfg.noise_std / (feat_std + 1e-8)
    print(f"Noise σ={cfg.noise_std} → normalised σ≈{norm_noise:.3f}")
    A_dist = None
    for key in ("adjacency_matrix", "adj_mx", "adj"):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f"Adjacency from \"{key}\""); break
    if A_dist is None:
        A_dist = np.eye(N, dtype=np.float32)
    nonzero = A_dist[A_dist > 0]
    sigma   = nonzero.std() if len(nonzero) > 0 else 1.0
    A_dist  = np.exp(-(A_dist**2) / (sigma**2 + 1e-8))
    np.fill_diagonal(A_dist, 0.0)
    A_dist  = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
    return data_clean, mean_np, std_np, A_dist, norm_noise


class TrafficDataset(Dataset):
    """Returns (x_rec_noisy, x_hour_noisy, y_clean).
    FIX 1: inputs are noisy, labels are clean — paper's training setup.
    FIX 5: returns hourly sequence for multi-period modeling."""
    def __init__(self, data_clean, seq_len, pred_len, feature_idx,
                 hour_offset=12, norm_noise=0.0,
                 split_start=0, split_end=None, training=False):
        self.data        = data_clean
        self.seq_len     = seq_len
        self.pred_len    = pred_len
        self.feat_idx    = feature_idx
        self.hour_offset = hour_offset
        self.norm_noise  = norm_noise
        self.training    = training
        T = len(data_clean)
        split_end = split_end if split_end is not None else T
        # first valid i: hour lookback must not go below 0
        first_i = max(split_start, hour_offset)
        last_i  = split_end - seq_len - pred_len + 1
        self.indices = list(range(first_i, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i     = self.indices[idx]
        rec   = self.data[i : i + self.seq_len].copy()
        hour  = self.data[i - self.hour_offset :
                          i - self.hour_offset + self.seq_len].copy()
        y     = self.data[i + self.seq_len :
                          i + self.seq_len + self.pred_len,
                          :, self.feat_idx].copy()
        # Add noise to inputs ONLY — labels stay clean (paper setup)
        if self.training and self.norm_noise > 0:
            rec  += np.random.randn(*rec.shape ).astype(np.float32) * self.norm_noise
            hour += np.random.randn(*hour.shape).astype(np.float32) * self.norm_noise
        return (torch.from_numpy(rec),
                torch.from_numpy(hour),
                torch.from_numpy(y))


def build_dataloaders(cfg):
    set_seed()
    data_clean, mean_np, std_np, A_dist, norm_noise = load_pems08(cfg)
    T  = len(data_clean)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=0, pin_memory=False)
    ds_kw = dict(seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx, hour_offset=cfg.hour_offset,
                 norm_noise=norm_noise, data_clean=data_clean)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1, training=True),  shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2, training=False), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T,  training=False), shuffle=False, **kw)
    print(f"Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}")
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist, norm_noise

print("Data utilities ready.")


Data utilities ready.


In [4]:
class InputProjection(nn.Module):
    """Projects single sequence in_features → d_model."""
    def __init__(self, in_dim, d_model, dropout=0.1):
        super().__init__()
        self.proj  = nn.Linear(in_dim, d_model)
        self.norm  = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        h = self.norm(self.drop(self.proj(x)))
        return self.norm2(h + self.ff(h))


class MultiPeriodFusion(nn.Module):
    """FIX 5: Fuse recent + hourly sequences via attention (paper MAF Eq.5-9).
    Two BackNets project independently, then fused via concat+linear."""
    def __init__(self, in_dim, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.proj_rec  = InputProjection(in_dim, d_model, dropout)
        self.proj_hour = InputProjection(in_dim, d_model, dropout)
        self.attn      = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.norm      = nn.LayerNorm(d_model)
        self.fuse      = nn.Linear(d_model * 2, d_model)

    def forward(self, x_rec, x_hour):
        # x_rec, x_hour: (B, S, N, F)
        B, S, N, F = x_rec.shape
        h_rec  = self.proj_rec (x_rec )   # (B,S,N,d)
        h_hour = self.proj_hour(x_hour)   # (B,S,N,d)
        # Cross-attention: recent queries hourly context
        d = h_rec.shape[-1]
        q = h_rec.reshape(B*S, N, d)
        k = h_hour.reshape(B*S, N, d)
        v = h_hour.reshape(B*S, N, d)
        att, _ = self.attn(q, k, v)
        att = self.norm(q + att).reshape(B, S, N, d)
        # Fuse: recent + attended-hourly
        fused = self.fuse(torch.cat([h_rec, att], dim=-1))
        return fused   # (B, S, N, d)

print("MultiPeriodFusion defined (recent + hourly).")


InputProjection + AdjacencyFusion defined.


In [5]:
class GraphAttention(nn.Module):
    """Multi-head GAT using the fused adjacency as an attention mask."""
    def __init__(self, in_dim, out_dim, n_heads=4, dropout=0.1):
        super().__init__()
        assert out_dim % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = out_dim // n_heads
        self.W       = nn.Linear(in_dim, out_dim, bias=False)
        # per-head attention vectors (size = d_head each)
        self.a_src   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.a_dst   = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.drop    = nn.Dropout(dropout)
        self.out     = nn.Linear(out_dim, out_dim)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, x, A):
        # x: (B, N, in_dim)   A: (N, N)
        B, N, _ = x.shape
        h = self.W(x).reshape(B, N, self.n_heads, self.d_head)  # (B,N,H,d)
        h = h.permute(0, 2, 1, 3)                                # (B,H,N,d)
        # e_src[b,h,i,1] = (h[b,h,i,:] * a_src).sum(-1)
        e_src = (h * self.a_src).sum(-1, keepdim=True)           # (B,H,N,1)
        # e_dst[b,h,1,j] = (h[b,h,j,:] * a_dst).sum(-1)
        e_dst = (h * self.a_dst).sum(-1, keepdim=True)           # (B,H,N,1)
        # broadcast: (B,H,N,1) + (B,H,1,N) → (B,H,N,N)
        e = F.leaky_relu(e_src + e_dst.permute(0, 1, 3, 2), negative_slope=0.2)
        # mask non-edges
        mask  = (A == 0).unsqueeze(0).unsqueeze(0)               # (1,1,N,N)
        e     = e.masked_fill(mask, -1e9)
        alpha = self.drop(torch.softmax(e, dim=-1))               # (B,H,N,N)
        out   = (alpha @ h).permute(0, 2, 1, 3).reshape(B, N, -1)  # (B,N,out_dim)
        return self.out(out)


class GATLayer(nn.Module):
    """Single GAT layer with residual + LayerNorm.
    Applied independently to each timestep: (B,N,d) → (B,N,d).
    No GRU — temporal dependencies handled by STFormer."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.gat  = GraphAttention(d_model, d_model, n_heads, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):
        # x: (B, N, d)   A: (N, N)
        x = self.norm (x + self.drop(self.gat(x, A)))   # GAT + residual
        x = self.norm2(x + self.drop(self.ff(x)))        # FFN + residual
        return x


class MGRCModule(nn.Module):
    """Pure stacked GAT (no GRU, no attention beyond GAT).
    Each layer: fused-adj GAT → residual → LayerNorm → FFN → residual → LayerNorm.
    Applied across all timesteps independently — temporal context from STFormer."""
    def __init__(self, in_dim, hidden_dim, num_nodes, num_layers=4, n_heads=4, dropout=0.1):
        super().__init__()
        # Dynamic adjacency embedding (paper Eq.10)
        self.emb      = nn.Parameter(torch.randn(num_nodes, hidden_dim))
        self.adj_conv = nn.Conv2d(2, 1, kernel_size=1)
        # Input projection if dims differ
        self.input_proj = nn.Linear(in_dim, hidden_dim) if in_dim != hidden_dim else nn.Identity()
        # Stacked GAT layers — all same dimension
        self.layers = nn.ModuleList([
            GATLayer(hidden_dim, n_heads, dropout) for _ in range(num_layers)])

    def get_fused_adj(self, A_dist):
        A_dyna  = torch.softmax(self.emb @ self.emb.T, dim=-1)
        stacked = torch.stack([A_dist, A_dyna], dim=0).unsqueeze(0)  # (1,2,N,N)
        A_F     = F.relu(self.adj_conv(stacked).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_dist):
        # x: (B, S, N, d)
        B, S, N, d = x.shape
        A_F = self.get_fused_adj(A_dist)              # (N, N)
        x   = self.input_proj(x)                       # (B, S, N, hidden)
        # Apply each GAT layer across all timesteps independently
        x_flat = x.reshape(B*S, N, -1)                # (B*S, N, d)
        for layer in self.layers:
            x_flat = layer(x_flat, A_F)
        return x_flat.reshape(B, S, N, -1)             # (B, S, N, d)

print("MGRC defined — pure stacked GAT (no GRU).")

MGRC defined — pure stacked GAT (no GRU).


In [6]:
class SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding (LLM-style, not learnable).
    Provides strong inductive bias for sequence order."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d)

    def forward(self, x):
        # x: (B*N, S, d)
        return x + self.pe[:, :x.size(1)]


class TemporalTransformerLayer(nn.Module):
    """Temporal transformer with LLM-style sinusoidal PE + learnable TPE."""
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        # LLM-style fixed sinusoidal PE
        self.sin_pe = SinusoidalPE(d_model)
        # Learnable hourly/daily/weekly weights (paper Eq.21)
        self.W_hour = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_day  = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_week = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        t = torch.arange(seq_len).float()
        self.register_buffer("E_hour", (t % 12 + 1).unsqueeze(0))
        self.register_buffer("E_day",  (t % 24 + 1).unsqueeze(0))
        self.register_buffer("E_week", (t % 7  + 1).unsqueeze(0))
        self.tpe_proj = nn.Linear(1, d_model)
        # Transformer block
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(),
                                   nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, S, N, d)
        B, S, N, d = x.shape
        # Learnable temporal PE
        enc = (self.W_hour @ self.E_hour +
               self.W_day  @ self.E_day  +
               self.W_week @ self.E_week)         # (N, S)
        enc = self.tpe_proj(enc.T.unsqueeze(0).unsqueeze(-1))  # (1, S, N, d)
        x   = x + enc
        # Reshape for temporal attention
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        # LLM-style sinusoidal PE
        f = self.sin_pe(f)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print("Temporal transformer with sinusoidal PE defined.")


Temporal transformer with sinusoidal PE defined.


In [7]:
class MDGRTN(nn.Module):
    """
    Paper-aligned architecture:
    (x_rec_noisy, x_hour_noisy) → MultiPeriodFusion → GAT stack → Temporal Transformer → Output
    Noisy input, clean output — paper's core training paradigm.
    """
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        h, dr   = cfg.n_heads, cfg.dropout
        P, S    = cfg.pred_len, cfg.seq_len

        # FIX 5: Multi-period fusion of recent + hourly sequences
        self.mpf = MultiPeriodFusion(F, d, h, dr)

        # GAT spatial stack (no redundant AdjacencyFusion — FIX 7)
        self.gat_stack = MGRCModule(d, d, N, num_layers=cfg.gat_layers,
                                    n_heads=h, dropout=dr)

        self.temp_layers = nn.ModuleList([
            TemporalTransformerLayer(d, h, dr, S, N)
            for _ in range(cfg.temp_layers)])

        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x_rec, x_hour, A_dist):
        x = self.mpf(x_rec, x_hour)         # (B,S,N,d)
        x = self.gat_stack(x, A_dist)        # spatial
        for layer in self.temp_layers:       # temporal
            x = layer(x)
        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)


set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready | Parameters: {total:,}")
print(f"  d={cfg.d_model} | GAT={cfg.gat_layers} | Temp={cfg.temp_layers}")
with torch.no_grad():
    dummy_r = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    dummy_h = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj     = torch.eye(cfg.num_nodes).to(device)
    out     = model(dummy_r, dummy_h, adj)
print(f"Output shape: {out.shape}  ✓")


Seed set: 42 — results reproducible across sessions ✓
Model ready | Parameters: 872,229
  d=96 | GAT=4 | Temp=5 | dropout=0.25
Output shape: torch.Size([2, 12, 170])  ✓


In [8]:
def masked_mae(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return (torch.abs(pred-true)*mask).sum() / (mask.sum()+1e-8)

def huber_loss(pred, true, delta=1.0,  # FIX 3: paper standard null_val=0.0):
    """Smooth L1 / Huber — less sensitive to outliers than MAE."""
    mask = (true != null_val).float()
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5 * err**2, delta * (err - 0.5 * delta))
    return (loss * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return torch.sqrt(((pred-true)**2*mask).sum() / (mask.sum()+1e-8))

def masked_mape(pred, true, low_thresh=10.0):
    """Mask near-zero flow to avoid div/0."""
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics defined.')

Metrics defined.


In [9]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np, norm_noise = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)
print(f"Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}")


Seed set: 42 — results reproducible across sessions ✓
Shape: (17856, 170, 3)
Noise disabled — clean inputs for both train and eval
Train=12470 | Val=1756 | Test=3543
Train batches: 195 | Val: 28 | Test: 56


In [10]:
scaler = torch.amp.GradScaler("cuda")

def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, x_hour, A_dist)
            loss = huber_loss(pred, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, x_hour, A_dist)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print("Train/eval defined.")


Train/eval with mixed precision (AMP) defined.


In [11]:
# Simple checkpoint save — no gru_layers reference
def save_best(model, path):
    torch.save(model.state_dict(), path)
    print(f"Best model saved → {path}")

print("Checkpoint utility ready.")


Checkpoint utility ready.


In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                               weight_decay=cfg.weight_decay)
# FIX 6: ReduceLROnPlateau responds to actual val MAE
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=8, factor=0.5, min_lr=1e-5)

best_val_mae = float("inf")
patience_cnt = 0
history      = {"train_loss":[], "val_mae":[], "val_rmse":[], "val_mape":[]}

print(f"Training | d={cfg.d_model} | GAT={cfg.gat_layers} | Temp={cfg.temp_layers}")
print(f"lr={cfg.lr} | wd={cfg.weight_decay} | noise_std={cfg.noise_std} | batch={cfg.batch_size}")
print(f"Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%")
print("="*65)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step(val_mae)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_mape"].append(val_mape)

    tag = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        save_best(model, cfg.best_path)
        tag = "  ← best ✓"
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    lr_now = optimizer.param_groups[0]["lr"]
    if epoch % 5 == 0 or tag:
        print(f"Ep {epoch:03d} | Loss={train_loss:.4f} | "
              f"MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% "
              f"lr={lr_now:.1e}{tag}")

print(f"\nBest Val MAE: {best_val_mae:.3f}")


Seed set: 42 — results reproducible across sessions ✓
Training | d=96 | GAT=4 | Temp=5
lr=0.0005 | wd=0.0001 | dropout=0.25 | batch=64
Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%
Best model saved → mdgrtn_best.pt
Ep 001 | Loss=0.0698 | MAE=20.334 RMSE=30.828 MAPE=11.42% lr=5.0e-04  ← best ✓
Best model saved → mdgrtn_best.pt
Ep 002 | Loss=0.0413 | MAE=19.066 RMSE=29.070 MAPE=10.66% lr=5.0e-04  ← best ✓
Best model saved → mdgrtn_best.pt
Ep 003 | Loss=0.0377 | MAE=18.045 RMSE=27.724 MAPE=10.30% lr=5.0e-04  ← best ✓
Best model saved → mdgrtn_best.pt
Ep 005 | Loss=0.0348 | MAE=17.281 RMSE=26.665 MAPE=9.75% lr=5.0e-04  ← best ✓
Best model saved → mdgrtn_best.pt
Ep 008 | Loss=0.0320 | MAE=16.701 RMSE=25.938 MAPE=9.39% lr=5.0e-04  ← best ✓
Ep 010 | Loss=0.0308 | MAE=17.045 RMSE=26.409 MAPE=9.75% lr=5.0e-04
Best model saved → mdgrtn_best.pt
Ep 015 | Loss=0.0293 | MAE=16.417 RMSE=25.605 MAPE=9.27% lr=4.9e-04  ← best ✓
Best model saved → mdgrtn_best.pt
Ep 018 | Loss=0.0283 | MAE=16.091 RMSE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# FINAL TEST — paper-style single averaged metric
# ══════════════════════════════════════════════════
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*55)
    print('  TEST RESULTS  —  averaged over all 12 steps')
    print('='*55)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*55)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(
    model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)